In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Your CRUD Python module from Project One (file: CRUD_Python_Module.py).
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Credentials hard coded here per the Project Two instructions.
username = "aacuser"
password = "SNHU1234"   # confirm this is the password you created for aacuser

# Connect to database via your CRUD Module. Host/port live inside
# CRUD_Python_Module.py (the connection you already set in Project One).
db = AnimalShelter(username, password)

# Unique identifier required by the spec.
developer_name = "Eric Shadwick"

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True)


###########################################################
# Filter queries (grounded in the Rescue Type breed table)
###########################################################
# Breeds are matched with a case-insensitive regex CONTAINS because the breed
# field is full of compound strings (e.g. "Labrador Retriever/Newfoundland").
# An exact match list would miss almost every mixed breed.
def build_query(filter_type):
    if filter_type == 'water':
        return {
            "breed": {"$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland",
                      "$options": "i"},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0},
        }
    if filter_type == 'mountain':
        return {
            "breed": {"$regex": ("German Shepherd|Alaskan Malamute|Old English Sheepdog|"
                                 "Siberian Husky|Rottweiler"), "$options": "i"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0},
        }
    if filter_type == 'disaster':
        return {
            "breed": {"$regex": ("Doberman Pinscher|German Shepherd|Golden Retriever|"
                                 "Bloodhound|Rottweiler"), "$options": "i"},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0},
        }
    return {}   # reset / unfiltered


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo. Filename has spaces, matching your Codio file tree.
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    # Logo anchored to the SNHU home page, plus the unique identifier.
    html.Center(html.A(
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 style={'height': '200px'}),
        href='https://www.snhu.edu', target='_blank')),
    html.Center(html.B('Created by ' + developer_name)),
    html.Hr(),

    # Interactive filtering options (radio items).
    html.Div(
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'},
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        column_selectable="single",
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px'},
    ),
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6', style={'width': '50%'}),
                 html.Div(id='map-id', className='col s12 m6', style={'width': '50%'}),
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    dff = pd.DataFrame.from_records(db.read(build_query(filter_type)))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff.to_dict('records')


# Pie chart of breed distribution for whatever the table currently shows.
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty or 'breed' not in dff.columns:
        return []
    # Collapse the long tail so the unfiltered view is readable: keep the top
    # 10 breeds, roll everything else into a single "Other Breeds" slice.
    counts = dff['breed'].value_counts()
    top = counts.head(10)
    labels = list(top.index)
    values = [int(v) for v in top.values]
    other = int(counts.iloc[10:].sum())
    if other > 0:
        labels.append('Other Breeds')
        values.append(other)
    fig = px.pie(names=labels, values=values, title='Preferred Breeds')
    fig.update_traces(textposition='inside')
    fig.update_layout(uniformtext_minsize=10, uniformtext_mode='hide')
    return [dcc.Graph(figure=fig)]


# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None:
        return []
    dff = pd.DataFrame.from_dict(viewData)
    if dff.empty:
        return []
    row = index[0] if index else 0
    if row >= len(dff):
        row = 0
    # Columns pulled by NAME, not positional index (iloc[row,13] etc.), so the
    # map does not break if column order changes.
    lat   = dff.iloc[row]['location_lat']
    lon   = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name  = dff.iloc[row]['name']
    # Austin TX is at [30.75, -97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(name))
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app,
# the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()


Dash app running on https://vocalbamboo-cyclonenoise-3000.codio.io/proxy/8050/
